In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/priths24/dataset/dataset_variable_description.xlsx
/kaggle/input/datasets/priths24/dataset/test-dataset.xlsx


In [2]:
!pip install -q openpyxl
import pandas, numpy, scipy, openpyxl
print(pandas.__version__, numpy.__version__, scipy.__version__, openpyxl.__version__)

2.3.3 2.0.2 1.16.3 3.1.5


In [3]:
# Cell 2
import glob, shutil, os
matches = glob.glob('/kaggle/input/**/*test*dataset*.xlsx', recursive=True)
print(matches)          # expect one path ending in test-dataset.xlsx
shutil.copy(matches[0], '/kaggle/working/test-dataset.xlsx')
os.chdir('/kaggle/working')

['/kaggle/input/datasets/priths24/dataset/test-dataset.xlsx']


In [4]:
"""
AIMS Lab Technical Assessment - Task 2
Which features are statistically associated with hypertension?

Outcome : profile_hypertensive (recorded hypertension status, True/False)
Tests   : Pearson chi-square test of independence (categorical features)
          Fisher's exact test (2x2 tables with any expected count < 5)
          Mann-Whitney U test (continuous features)
Extras  : Cramer's V (effect size), Bonferroni correction (multiple testing)

Run: python analysis.py   (needs pandas, numpy, scipy, openpyxl)
"""

import numpy as np
import pandas as pd
from scipy import stats

ALPHA = 0.05

# --------------------------------------------------------------------------
# 1. Load data
# --------------------------------------------------------------------------
df = pd.read_excel("test-dataset.xlsx")
n_raw = len(df)

# --------------------------------------------------------------------------
# 2. Preprocessing
# --------------------------------------------------------------------------
# 2a. Drop personal identifiers (privacy) and non-informative columns
df = df.drop(columns=["Unnamed: 0", "profile_name", "father_name",
                      "mother_name", "birthday", "user_id"])
# is_poor is 0 for every row -> constant, carries no information
assert df["is_poor"].nunique() == 1
df = df.drop(columns=["is_poor"])

# 2b. Adults only: no one under 18 is recorded as hypertensive
n_minors = (df["age"] < 18).sum()
df = df[(df["age"] >= 18) & (df["age"] <= 100)].copy()   # >100 treated as entry error

# 2c. Physiologically implausible readings -> set to missing (not dropping rows)
def clean(col, lo, hi):
    bad = ~df[col].between(lo, hi) & df[col].notna()
    df.loc[bad, col] = np.nan
    return int(bad.sum())

n_bad = {
    "SYSTOLIC":   clean("SYSTOLIC", 70, 250),
    "DIASTOLIC":  clean("DIASTOLIC", 40, 150),
    "PULSE_RATE": clean("PULSE_RATE", 30, 200),
    "SPO2":       clean("SPO2", 70, 100),
    "BMI":        clean("BMI", 10, 60),
    "SUGAR":      clean("SUGAR", 1, 35),
}
# systolic must exceed diastolic
bad_bp = (df["SYSTOLIC"] <= df["DIASTOLIC"])
df.loc[bad_bp, ["SYSTOLIC", "DIASTOLIC"]] = np.nan
n_bad["SBP<=DBP"] = int(bad_bp.sum())

# 2d. Derived / regrouped categorical features
df["htn"] = df["profile_hypertensive"].astype(bool)
df["age_group"] = pd.cut(df["age"], [17, 29, 44, 59, 100],
                         labels=["18-29", "30-44", "45-59", "60+"])
df["disability"] = df["disabilities_name"].astype(str) != "0"
df["spo2_status"] = df["RESULT_STAT_SPO2"].replace({"Very low": "Low"})
df["bmi_status"] = df["RESULT_STAT_BMI"].replace(
    {"Highly Obesity": "Obese", "Morbid Obesity": "Obese", "Obesity": "Obese"})
df["overweight_obese"] = df["bmi_status"].map(
    lambda s: np.nan if pd.isna(s) else s in ("Overweight", "Obese"))
df["sugar_elevated"] = df["RESULT_STAT_SUGAR"].map(
    lambda s: np.nan if pd.isna(s) else s not in ("Normal", "Low", "LOW (Hypoglycemia)"))

# Secondary outcome: measured high BP at screening (>=140/90), objective reading
df["high_bp"] = np.where(df["SYSTOLIC"].notna() & df["DIASTOLIC"].notna(),
                         ((df["SYSTOLIC"] >= 140) | (df["DIASTOLIC"] >= 90)).astype(float),
                         np.nan)

# --------------------------------------------------------------------------
# 3. Tests
# --------------------------------------------------------------------------
def cramers_v(chi2, n, r, c):
    return np.sqrt(chi2 / (n * (min(r, c) - 1)))

def categorical_test(feature, label, outcome="htn"):
    """Chi-square test of independence, or Fisher's exact test for sparse 2x2."""
    sub = df[[feature, outcome]].dropna()
    table = pd.crosstab(sub[feature], sub[outcome].astype(bool))
    table = table.loc[table.sum(axis=1) > 0]
    chi2, p, dof, expected = stats.chi2_contingency(table, correction=False)
    n = table.values.sum()
    v = cramers_v(chi2, n, *table.shape)
    test = "Chi-square"
    if table.shape == (2, 2) and (expected < 5).any():
        _, p = stats.fisher_exact(table)
        test = "Fisher"
    prev = (table[True] / table.sum(axis=1) * 100).round(1)
    pattern = "; ".join(f"{k}: {val}%" for k, val in prev.items())
    return dict(feature=label, test=test, n=n, stat=chi2, df=dof, p=p,
                effect=v, pattern=pattern, min_expected=expected.min())

def continuous_test(feature, label, outcome="htn"):
    """Mann-Whitney U (distribution-free) + rank-biserial correlation."""
    sub = df[[feature, outcome]].dropna()
    y = sub[outcome].astype(bool)
    a, b = sub.loc[y, feature], sub.loc[~y, feature]
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    r_rb = 1 - 2 * u / (len(a) * len(b))        # rank-biserial effect size
    pattern = (f"HTN median {a.median():.1f} (IQR {a.quantile(.25):.1f}-{a.quantile(.75):.1f})"
               f" vs {b.median():.1f} ({b.quantile(.25):.1f}-{b.quantile(.75):.1f})")
    return dict(feature=label, test="Mann-Whitney", n=len(sub), stat=u, df=np.nan,
                p=p, effect=abs(r_rb), pattern=pattern, min_expected=np.nan)

cat_features = [
    ("age_group", "Age group"),
    ("gender", "Gender"),
    ("total_income", "Income class"),
    ("union_name", "Union (16)"),
    ("diabetic", "Diabetes (recorded)"),
    ("had_stroke", "Stroke history"),
    ("has_cardiovascular_disease", "CVD history"),
    ("disability", "Any disability"),
    ("is_freedom_fighter", "Freedom fighter"),
    ("RESULT_STAT_PR", "Pulse status"),
    ("spo2_status", "SpO2 status"),
    ("overweight_obese", "Overweight/obese"),
    ("sugar_elevated", "Elevated sugar"),
]
cont_features = [
    ("age", "Age (years)"),
    ("PULSE_RATE", "Pulse rate"),
    ("SPO2", "SpO2 (%)"),
    ("BMI", "BMI"),
    ("SUGAR", "Blood sugar"),
]

rows = [categorical_test(f, l) for f, l in cat_features] + \
       [continuous_test(f, l) for f, l in cont_features]
res = pd.DataFrame(rows)

# Bonferroni correction across all tests
m = len(res)
res["p_bonf"] = np.minimum(res["p"] * m, 1.0)
res["significant"] = res["p_bonf"] < ALPHA

# Sensitivity analysis: same tests against measured high BP
rows2 = [categorical_test(f, l, "high_bp") for f, l in cat_features] + \
        [continuous_test(f, l, "high_bp") for f, l in cont_features]
res2 = pd.DataFrame(rows2)
res2["p_bonf"] = np.minimum(res2["p"] * m, 1.0)
res2["significant"] = res2["p_bonf"] < ALPHA

# --------------------------------------------------------------------------
# 4. Worked example: gender x hypertension (2x2), by hand
# --------------------------------------------------------------------------
t = pd.crosstab(df["gender"], df["htn"])
O = t.values
R, C, N = O.sum(axis=1), O.sum(axis=0), O.sum()
E = np.outer(R, C) / N
chi2_hand = ((O - E) ** 2 / E).sum()
chi2_lib = stats.chi2_contingency(O, correction=False)[0]
assert np.isclose(chi2_hand, chi2_lib)
odds_ratio = (O[0, 1] * O[1, 0]) / (O[0, 0] * O[1, 1])   # Female vs Male

# --------------------------------------------------------------------------
# 5. Analytic-outcome figures
# --------------------------------------------------------------------------
bp = df.dropna(subset=["SYSTOLIC", "DIASTOLIC"])
high_bp = (bp["SYSTOLIC"] >= 140) | (bp["DIASTOLIC"] >= 90)
undiag = bp[~bp.htn]
undetected = ((undiag["SYSTOLIC"] >= 140) | (undiag["DIASTOLIC"] >= 90))
union_prev = df.groupby("union_name")["htn"].mean().mul(100).sort_values()

# --------------------------------------------------------------------------
# 6. Output
# --------------------------------------------------------------------------
pd.set_option("display.width", 250, "display.max_colwidth", 120)
print(f"Rows: {n_raw} raw -> {len(df)} adults (18-100); minors removed: {n_minors}")
print("Implausible values set to missing:", n_bad)
print(f"Hypertension prevalence (adults): {df.htn.mean()*100:.1f}% ({df.htn.sum()})")
print(f"Tests: {m}; Bonferroni alpha = {ALPHA/m:.5f}\n")
out = res[["feature", "test", "n", "stat", "df", "p", "p_bonf", "effect",
           "significant", "min_expected", "pattern"]]
print(out.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\nWorked example (gender x HTN):")
print(t)
print("Expected:\n", np.round(E, 1))
print(f"chi2 = {chi2_hand:.2f}, df = 1, OR(F vs M) = {odds_ratio:.2f}, "
      f"V = {cramers_v(chi2_hand, N, 2, 2):.3f}")
print(f"\nAdults with a BP reading: {len(bp)}; high BP (>=140/90): {high_bp.mean()*100:.1f}%")
print(f"Undetected: {undetected.sum()} of {len(undiag)} "
      f"({undetected.mean()*100:.1f}%) not recorded as hypertensive")
print("Union HTN prevalence range: "
      f"{union_prev.min():.1f}% ({union_prev.idxmin()}) to "
      f"{union_prev.max():.1f}% ({union_prev.idxmax()})")
print("\nSensitivity: measured high BP (>=140/90) as outcome")
print(res2[["feature","test","n","stat","df","p_bonf","effect","significant","pattern"]]
      .to_string(index=False, float_format=lambda x: f"{x:.4g}"))
union_cmp = df.groupby("union_name").agg(recorded=("htn","mean"), measured=("high_bp","mean")).mul(100).round(1)
print("\nUnion: recorded status vs measured high BP (%)")
print(union_cmp.sort_values("recorded").to_string())
res.to_csv("task2_results.csv", index=False)
res2.to_csv("task2_results_measured_bp.csv", index=False)

Rows: 29999 raw -> 27133 adults (18-100); minors removed: 2726
Implausible values set to missing: {'SYSTOLIC': 20, 'DIASTOLIC': 14, 'PULSE_RATE': 9, 'SPO2': 10, 'BMI': 4, 'SUGAR': 2, 'SBP<=DBP': 0}
Hypertension prevalence (adults): 5.8% (1575)
Tests: 18; Bonferroni alpha = 0.00278

            feature         test     n      stat  df          p     p_bonf   effect  significant  min_expected                                                                                                                                                                                                                                                                                pattern
          Age group   Chi-square 27133      1098   3 1.151e-237 2.071e-236   0.2011         True         214.9                                                                                                                                                                                                                         

In [5]:
import os
os.remove('/kaggle/working/test-dataset.xlsx')